# IEEE-CIS Fraud Detection — Modelling

## Goal

Train three gradient boosting models on the engineered features from notebook 02.
Each model is logged as a separate MLflow run so results are comparable.

## Why these three models?

| Model | Strength |
|---|---|
| XGBoost | Battle-tested, great default performance, wide community support |
| LightGBM | Fastest training, handles high cardinality well, good on large datasets |
| CatBoost | Best native categorical handling, less hyperparameter tuning needed |

All three are gradient boosting — ensemble of decision trees built sequentially,
each tree correcting the errors of the previous one.

## Evaluation Metric

**Primary: AUC-PR (Area Under Precision-Recall Curve)**

Not accuracy. Not AUC-ROC. AUC-PR because:
- 96.5% accuracy is achievable by predicting everything as legitimate — useless
- AUC-ROC is optimistic under class imbalance (28:1 ratio)
- AUC-PR focuses on the positive (fraud) class — more informative when fraud is rare

**Secondary: AUC-ROC, F1** — for completeness and comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report
)

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

# Load processed data
train_df = pd.read_parquet('data/processed/train_features.parquet')
val_df   = pd.read_parquet('data/processed/val_features.parquet')

with open('data/processed/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols]
y_train = train_df['isFraud']
X_val   = val_df[feature_cols]
y_val   = val_df['isFraud']

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'Val fraud rate:   {y_val.mean()*100:.2f}%')

## Chapter 1 — Class Imbalance Strategy

28:1 ratio means the model will see 28 legitimate transactions for every 1 fraud.
Without correction it will learn to predict everything as legitimate.

**Solution: `scale_pos_weight`**

Tells the model to penalise missed fraud more heavily by upweighting the positive class:

```
scale_pos_weight = count(negative) / count(positive)
                 = 569,877 / 20,663
                 ≈ 27.6
```

This makes the model treat each fraud case as if it were 27.6 legitimate cases —
balancing the gradient updates during training.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

## Chapter 2 — MLflow Setup

Each model is logged as a separate MLflow run under one experiment.
This lets you compare AUC-PR, AUC-ROC, F1 side by side in the MLflow UI.

We log:
- All hyperparameters
- AUC-PR, AUC-ROC, F1 scores
- Confusion matrix as an artifact
- Feature importance plot as an artifact
- The model itself (for later deployment)

In [ ]:
mlflow.set_experiment('fraud-detection')
print('MLflow experiment set: fraud-detection')

## Chapter 3 — XGBoost

Start with XGBoost — most familiar, good baseline.

Key parameters:
- `scale_pos_weight` — handles class imbalance
- `eval_metric='aucpr'` — optimises for AUC-PR during training
- `early_stopping_rounds=50` — stops if val AUC-PR doesn't improve for 50 rounds
- `n_estimators=1000` — high ceiling, early stopping will find the right number

### Your task

Fill in the blanks and run the XGBoost training cell below.

In [ ]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'aucpr',
    'early_stopping_rounds': 50,
    'random_state': 42,
    'n_jobs': -1
}

with mlflow.start_run(run_name='xgboost_baseline'):
    mlflow.log_params(xgb_params)
    
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    mlflow.sklearn.log_model(xgb_model, 'xgboost_model')
    
    print(f'XGBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')
    print(f'  Best iteration: {xgb_model.best_iteration}')

## Chapter 4 — LightGBM

LightGBM is faster than XGBoost on large datasets — uses leaf-wise tree growth
instead of level-wise, finding better splits faster.

Key difference from XGBoost:
- `is_unbalance=True` instead of `scale_pos_weight` — automatic balancing
- `metric='average_precision'` — equivalent to AUC-PR

In [ ]:
from lightgbm import LGBMClassifier

lgbm_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'is_unbalance': True,
    'metric': 'average_precision',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

with mlflow.start_run(run_name='lightgbm_baseline'):
    mlflow.log_params(lgbm_params)
    
    lgbm_model = LGBMClassifier(**lgbm_params)
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[]
    )
    
    y_pred_proba = lgbm_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    mlflow.sklearn.log_model(lgbm_model, 'lightgbm_model')
    
    print(f'LightGBM Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')

## Chapter 5 — CatBoost

CatBoost handles categorical features natively — no label encoding needed.
It builds symmetric trees and uses ordered boosting to reduce overfitting.

Key difference:
- `auto_class_weights='Balanced'` — automatic class weight calculation
- `cat_features` — pass categorical column indices directly, no encoding needed
- `eval_metric='PRAUC'` — AUC-PR in CatBoost notation

In [ ]:
from catboost import CatBoostClassifier

catboost_params = {
    'iterations': 1000,
    'depth': 6,
    'learning_rate': 0.05,
    'auto_class_weights': 'Balanced',
    'eval_metric': 'PRAUC',
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 100
}

with mlflow.start_run(run_name='catboost_baseline'):
    mlflow.log_params(catboost_params)
    
    cat_model = CatBoostClassifier(**catboost_params)
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val)
    )
    
    y_pred_proba = cat_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    mlflow.sklearn.log_model(cat_model, 'catboost_model')
    
    print(f'CatBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')

## Chapter 6 — Model Comparison

Compare all three models side by side and pick the best for threshold optimisation.

In [ ]:
results = {}

for name, model in [('XGBoost', xgb_model), ('LightGBM', lgbm_model), ('CatBoost', cat_model)]:
    y_prob = model.predict_proba(X_val)[:, 1]
    results[name] = {
        'AUC-PR':  round(average_precision_score(y_val, y_prob), 4),
        'AUC-ROC': round(roc_auc_score(y_val, y_prob), 4),
        'F1':      round(f1_score(y_val, (y_prob >= 0.5).astype(int)), 4)
    }

results_df = pd.DataFrame(results).T
print(results_df.sort_values('AUC-PR', ascending=False))

best_model_name = results_df['AUC-PR'].idxmax()
print(f'\nBest model: {best_model_name}')